# w13_cpu1_dair_roadside_neighbor_graph_o2
Inputs: `shortpose-wave1-private-inputs` + `shortpose-beijing-four-junction-external-forecasts`


In [ ]:
import hashlib, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir(), 'Run on Kaggle'
REV = '7baf064779c92df2cd8628585889bb93a80f2f8b'
with tempfile.TemporaryDirectory(prefix='w13_dg_') as ts:
    tmp = Path(ts)
    repo, meta_dir, out_dir = tmp / 'repo', tmp / 'pinns_meta', tmp / 'out'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'CUDA_VISIBLE_DEVICES': ''}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    dair_zip = next(MOUNT.rglob('dair_roadside_third_cohort.zip'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env, check=True)
    subprocess.run([sys.executable, '-u', '-m', 'scripts.run_dair_roadside_neighbor_graph_o2', '--repo-root', str(repo), '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz), '--joint-zip', str(joint_bin), '--dair-zip', str(dair_zip), '--out-dir', str(out_dir)], cwd=repo, env=env, check=True)
    dest = WORK / 'dair_roadside_neighbor_graph_o2_5seed.zip'
    shutil.copy2(out_dir / 'dair_roadside_neighbor_graph_o2_5seed.zip', dest)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
